In [ ]:
### This script is used to analyze the 2p data and corresponding treadmill behavior data (USING DFF INSTEAD OF SPIKES)
### JSY, 03/2025

### This script is used to analyze the 2p data and corresponding treadmill behavior data
### 1. Load the 2p data and treadmill behavior data
### 2. Align the 2p data and treadmill behavior data
### 3a. Smooth the deconvolved traces using a 250 ms Gaussian window
### 3b. Remove any time points where the treadmill behavior data is not available (i.e. when the animal is not moving)
### 4. Spatial discretization (divide the VR corridor into ~110 bins, each representing 1cm and assign each data point to its corresponding spatial bin)
### 5. Response profile (add all the spikes or average all the data points in each spatial bin to get the activity map)
### 6. Test for reliability for individual cells (calculate Pearson CC or cohen’s D)
### 7. Plotting activity of all responsive cells (cross validation – split trials in half)
### 8. Population vector cross-correlation 

In [ ]:
%cd "C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation"

**1. Load the 2p data and treadmill behavior data**

In [ ]:
import os
import re
import datetime
import numpy as np
from helper import TwoP, read_xml, time2float
import matplotlib.pyplot as plt

# File path to VRlog.txt and 2p data
filepath = r'D:\V1_SpatialModulation\2p\250206_JSY_JSY038_SpatialModulation\TSeries-02062025-1350-001'
VRfilepath = r"D:\V1_SpatialModulation\V1_SpatialMod_VRLog"
twoP_filename = "TSeries-02062025-1350-001"
VRlog_filename = "VRlog_JSY038_02062025_01-56-15.txt"
VRlog_path = os.path.join(VRfilepath, VRlog_filename)

# Extract animal ID and date from the VR_log_filename
match = re.match(r"VRlog_(JSY\d+)_(\d{8})_\d{2}-\d{2}-\d{2}\.txt", VRlog_filename)
if match:
    animal_id = match.group(1)
    date = match.group(2)
else:
    print("Filename format does not match the expected pattern.")

# Initialize dictionaries to store raw data
twoP_data = {}
VR_data = {}

# Load twoP data
raw_twop_data = TwoP(filepath, twoP_filename)

raw_twop_data.find_files()
twop_dict = raw_twop_data.calc_dFF()

twoP_data['sps'] = twop_dict['spikes_per_sec'].copy()
twoP_data['s2p_spks'] = twop_dict['s2p_spks'].copy()
twoP_data['dFF'] = twop_dict['norm_dFF'].copy()

numFrames = np.size(twoP_data['sps'], 1)

xml_path = os.path.join(filepath, f"{twoP_filename}.xml")
xml_dict = read_xml(xml_path)
t0 = xml_dict["t0"]
framerate = xml_dict["acq_Hz"]/2
abs_time = xml_dict["abs_time"]

twopT = np.zeros(np.size(abs_time, 0) - 1, dtype=datetime.datetime)
for rep, t in enumerate(abs_time[:-1]):
    twopT[rep] = t0 + datetime.timedelta(seconds=t)

twopT_float = time2float(twopT)
twoP_data['AbsoluteT'] = twopT

# # print the data type of twoP_data['twopT'[0]]
# print(type(twoP_data['AbsoluteT'][0]))


# Load VRlog
rawVR_data = []
with open(VRlog_path, "r") as file:
    lines = file.readlines()
    for line in lines[3:]:
        rawVR_data.append(line.strip().split("\t"))
        
# Extract VR data
VR_data['absoluteT'] = np.array([line[0] for line in rawVR_data])
VR_data['elapsedT'] = np.array([float(line[1]) for line in rawVR_data])
VR_data['event'] = np.array([line[2] for line in rawVR_data])
VR_data['location'] = np.array([float(line[3]) for line in rawVR_data])

# Find the index of the first 's' in VR_data['event']
start_index = np.where(VR_data['event'] == 's')[0][0]

# Erase all elements before the start_index in all VR_data
for key in VR_data.keys():
    VR_data[key] = VR_data[key][start_index:]

# # for every element of VR_data, print first value
# for key in VR_data:
#     print(VR_data[key][0])

print("first time value of VR is", VR_data['absoluteT'][0])
print("first time value of 2p is", twoP_data['AbsoluteT'][0] )

**2. Align the 2p data and treadmill behavior data**

In [ ]:
# Define absolute_t0 as the third element of VR_data['absoluteT'] -- with "s" for event type, which is the timestamp for 2p input trigger
VR_absolute_t = np.array([datetime.datetime.strptime(t, '%H.%M.%S.%f') for t in VR_data['absoluteT'][0:]])

# Calculate relative_t (time elapsed from absolute_t0)
VR_relative_t = np.array([(t - VR_absolute_t[0]).total_seconds() for t in VR_absolute_t])

# Add twoP_data['AbsoluteT'][0] to each timedelta object to get vrT
VR_relative_t_timedelta = np.array([datetime.timedelta(seconds=t) for t in VR_relative_t])
Aligned_Abs_vrT = twoP_data['AbsoluteT'][0] + VR_relative_t_timedelta

# Find the closest value in Aligned_Abs_vrT that is greater than twoP_data['AbsoluteT'][-1]
closest_value = Aligned_Abs_vrT[Aligned_Abs_vrT > twoP_data['AbsoluteT'][-1]][0]
closest_index = np.where(Aligned_Abs_vrT == closest_value)[0][0]

# print(closest_value)
# print(closest_index)

new_VR_data = {}
new_VR_data['AbsoluteT'] = np.array(Aligned_Abs_vrT)[:closest_index]
new_VR_data['RelativeT'] = VR_relative_t[:closest_index]
new_VR_data['event'] = VR_data['event'][:closest_index]
new_VR_data['location'] = VR_data['location'][:closest_index]

# Calculate relative time points for VR_data and twoP_data
twop_relativeT = twoP_data['AbsoluteT'] - twoP_data['AbsoluteT'][0]

# Convert to seconds
twop_relativeT = np.array([t.total_seconds() for t in twop_relativeT])
twoP_data['RelativeT'] = twop_relativeT

# print("start of 2p is", twoP_data['RelativeT'][0])
# print("start of VR is", new_VR_data['RelativeT'][0])

# print("end of 2p is", twoP_data['RelativeT'][-1])
# print("end of VR is", new_VR_data['RelativeT'][-1])

# Interpolate the location at twoP_data['RelativeT'] from new_VR_data['location'] at new_VR_data['RelativeT']
interpolated_location = np.interp(twoP_data['RelativeT'], new_VR_data['RelativeT'], new_VR_data['location'])
new_VR_data['interp_location'] = interpolated_location

# for all other keys in new_VR_data, cut off the last few elements to match the length of interpolated_location
for key in new_VR_data.keys():
    new_VR_data[key] = new_VR_data[key][:len(interpolated_location)]
    

# # Now interpolated_location contains the interpolated location values at twop_relativeT
# print(new_VR_data['location'].shape)
# print(twoP_data['RelativeT'].shape)
# print(interpolated_location.shape)

# Plot the interpolated location
plt.figure(figsize=(20, 5))
plt.plot(twoP_data['RelativeT'], new_VR_data['interp_location']+10, label="Interpolated Location", alpha=0.5)
plt.plot(new_VR_data['RelativeT'], new_VR_data['location'], label="Original Location", alpha=0.5)
plt.legend()
plt.show()

# print(interpolated_location[0:10])
# print(aligned_VR_data['aligned_location'][0:10])

# plt.figure(figsize=(20, 5))
# plt.plot(twoP_data['RelativeT'], twoP_data['dFF'][10, :])
# plt.show()

# plt.figure(figsize=(20, 5))
# plt.plot(twoP_data['RelativeT'], twoP_data['sps'][10, :])
# plt.show()

**3a. Smooth the deconvolved traces using a 250 ms Gaussian window**

In [ ]:
from helper import SpikeSmoothing

smoothed = SpikeSmoothing.smooth_spikes(twoP_data['sps'], framerate, window_ms=250)
twoP_data['smoothed_spks'] = smoothed
SpikeSmoothing.plot_comparison(twoP_data['smoothed_spks'], twoP_data['sps'], framerate)

smoothed_dFF = SpikeSmoothing.smooth_spikes(twoP_data['dFF'], framerate, window_ms=250)
twoP_data['smoothed_dFF'] = smoothed_dFF

**3b. Remove any time points where the treadmill behavior data is not available (i.e. when the animal is not moving)**

In [ ]:
from helper import BehavioralDataFiltering as DF

filtered_dFF_laps, filtered_location_laps, n_valid_laps = DF.process_data_with_trial_filtering(
    twoP_data['smoothed_dFF'], 
    new_VR_data['interp_location'],
    max_trial_duration_seconds=30,
    framerate=framerate
)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# min_length = len(new_VR_data['interp_location'])

# # Define the number of frames to keep at the beginning and end of each stationary period
# frames_to_keep = 5

# # Initialize a mask to keep track of which frames to retain
# mask = np.ones(min_length, dtype=bool)

# # Find stationary periods directly from the truncated location data
# # Use a smaller window (e.g. n=1) for more accurate detection of stationary points
# location_diff = np.abs(np.diff(new_VR_data['interp_location'], n=10))
# stationary_mask = location_diff < 1e-2
# stationary_indices = np.where(stationary_mask)[0]

# # print(stationary_indices)

# # plt.plot(new_VR_data['RelativeT'],new_VR_data['interp_location'])
# # plt.plot(new_VR_data['RelativeT'][stationary_indices],new_VR_data['interp_location'][stationary_indices],  'ro')
# # plt.xlabel('Location (cm)')
# # plt.ylabel('Time (s)')
# # plt.title('Stationary periods detected in location data')
# # plt.show()


# # Group stationary indices into contiguous periods
# from itertools import groupby
# from operator import itemgetter

# def group_consecutive(data):
#     """Group consecutive indices together"""
#     for k, g in groupby(enumerate(data), lambda x: x[0] - x[1]):
#         yield list(map(itemgetter(1), g))

# # Group the stationary indices into contiguous periods
# stationary_periods = list(group_consecutive(stationary_indices))

# print(f"Found {len(stationary_periods)} contiguous stationary periods")

# # Apply the masking logic for each stationary period
# for period in stationary_periods:
#     if len(period) > 2 * frames_to_keep:
#         # Keep the first and last 'frames_to_keep' frames of each period
#         # and remove the middle portion
#         start_idx = period[0]
#         end_idx = period[-1]
        
#         # Mark frames to be removed (middle portion of the stationary period)
#         mask[start_idx + frames_to_keep:end_idx - frames_to_keep + 1] = False

# # apply the mask to every element of new_VR_data and twoP_data
# trimmed_VR_data = {}
# trimmed_twoP_data = {}

# for key in new_VR_data.keys():
#     trimmed_VR_data[key] = new_VR_data[key][mask]
    
# for key in twoP_data.keys():
#     if key in ['AbsoluteT', 'RelativeT']:
#         trimmed_twoP_data[key] = twoP_data[key][mask]
#     else:
#         trimmed_twoP_data[key] = twoP_data[key][:, mask]

# # Print final shapes to verify
# print("\nBefore shapes:")
# print(f"location: {new_VR_data['interp_location'].shape}")
# print(f"new_spks: {twoP_data['sps'].shape}")
# print(f"new_dFF: {twoP_data['dFF'].shape}")
# print(f"Number of time points kept: {np.sum(mask)} out of {min_length} ({np.sum(mask)/min_length*100:.1f}%)")

# print("\nFinal shapes:")
# print(f"location: {trimmed_VR_data['interp_location'].shape}")
# print(f"new_spks: {trimmed_twoP_data['sps'].shape}")
# print(f"new_dFF: {trimmed_twoP_data['dFF'].shape}")
# print(f"new_smoothedsps: {trimmed_twoP_data['smoothed_spks'].shape}")
# print(f"Number of time points kept: {np.sum(mask)} out of {min_length} ({np.sum(mask)/min_length*100:.1f}%)")


# # Plot to verify
# plt.figure(figsize=(12, 8))

# # plt.subplot(211)
# # plt.plot(trimmed_VR_data['interp_location'])
# # plt.title('New Location Data - inactive periods removed')
# # plt.xlabel('Time (frames)')
# # plt.ylabel('Location (cm)')

# # plt.subplot(212)
# # plt.plot(trimmed_twoP_data['dFF'][10,:])  # Plotting cell #10 as an example
# # plt.title('New dFF Data (example cell) - inactive periods removed')
# # plt.xlabel('Time (frames)')
# # plt.ylabel('dFF')

# # plt.tight_layout()
# # plt.show()

# # Optionally, visualize which frames were kept/removed
# plt.figure(figsize=(15, 8))

# # Plot location data
# plt.subplot(211)
# plt.plot(new_VR_data['interp_location'], 'b-', alpha=0.3, label='Original')
# plt.plot(np.arange(len(new_VR_data['interp_location']))[mask], 
#          new_VR_data['interp_location'][mask], 'r.', label='Kept')
# plt.title('Visualization of kept vs. removed frames - Location data')
# plt.legend()

# # Plot dFF data (example cell)
# plt.subplot(212)
# plt.plot(twoP_data['dFF'][10, :], 'g-', alpha=0.3, label='Original dFF')
# plt.plot(np.arange(len(twoP_data['dFF'][10, :]))[mask], 
#          twoP_data['dFF'][10, :][mask], 'm.', label='Kept dFF')
# plt.title('Visualization of kept vs. removed frames - dFF data (example cell)')
# plt.legend()

# plt.tight_layout()
# plt.show()

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(twoP_data['RelativeT'], interpolated_location, label="Interpolated Location", alpha=0.5)

# Highlight the stationary points (when animal was inactive -- movement less than 0.001 -- for 30 frames or 3 seconds)
stationary_indices = np.where(np.abs(np.diff(interpolated_location, n=50)) < 1e-2)[0]
for idx in stationary_indices:
    start_idx = idx
    # Find the end index where the location changes again
    change_indices = np.where(np.abs(np.diff(interpolated_location[start_idx:])) > 1e-2)[0]
    if change_indices.size > 0:
        end_idx = start_idx + change_indices[0]
        plt.axvspan(twoP_data['RelativeT'][start_idx], twoP_data['RelativeT'][end_idx], color='yellow', alpha=0.5)
    else:
        plt.axvspan(twoP_data['RelativeT'][start_idx], twoP_data['RelativeT'][-1], color='yellow', alpha=0.5)


plt.xlabel('Elapsed Time (s)')
plt.ylabel('Location')
plt.title('Location vs. Elapsed Time with Stationary Periods Highlighted')
plt.legend()
plt.show()

plt.figure(figsize=(20, 5))
plt.plot(twoP_data['RelativeT'], twoP_data['dFF'][10,:], label="dFF", alpha=0.5)
plt.title('dFF vs. Elapsed Time - example cell with activities')
plt.xlabel('Elapsed Time (s)')
plt.ylabel('dFF')
plt.legend()
plt.show()


plt.figure(figsize=(20, 5))
plt.plot(twoP_data['RelativeT'], twoP_data['sps'][10,:], label="dFF", alpha=0.5)
plt.title('dFF vs. Elapsed Time - example cell with activities')
plt.xlabel('Elapsed Time (s)')
plt.ylabel('spikes')
plt.legend()
plt.show()

# First, find the minimum length between VR and 2P data
# min_length = min(len(VR_data['location']), twoP_data['dFF'].shape[1])
min_length = min(len(VR_data['location']), twoP_data['smoothed_spks'].shape[1])
# print(f"Original lengths - VR: {len(VR_data['location'])}, 2P: {twoP_data['smoothed_spks'].shape[1]}")
# print(f"Using first {min_length} frames")

# Truncate the VR data to this length
VR_data_truncated = {'location': VR_data['location'][:min_length],'elapsedT': VR_data['elapsedT'][:min_length]}

# Truncate the 2P data
# twoP_data_truncated = twoP_data['dFF'][:, :min_length]
twoP_data_truncated = twoP_data['smoothed_spks'][:, :min_length]
dFF_data_truncated = twoP_data['dFF'][:, :min_length]

# Define the number of frames to keep at the beginning and end of each stationary period
frames_to_keep = 5

# Initialize a mask to keep track of which frames to retain
mask = np.ones(min_length, dtype=bool)

# Iterate over the stationary indices to update the mask
for idx in stationary_indices:
    if idx >= min_length:
        continue  # Skip if the index is beyond our truncated data
        
    start_idx = idx
    # Find the end index where the location changes again
    change_indices = np.where(np.abs(np.diff(VR_data_truncated['location'][start_idx:])) > 1e-2)[0]
    if change_indices.size > 0:
        end_idx = min(start_idx + change_indices[0], min_length)
    else:
        end_idx = min_length - 1

    # Mark the frames to be removed in the middle of the stationary period
    if end_idx - start_idx > 2 * frames_to_keep:
        mask[start_idx + frames_to_keep:end_idx - frames_to_keep] = False

# Apply the mask to create the new arrays
new_location = VR_data_truncated['location'][mask]
new_elapsedT = VR_data_truncated['elapsedT'][mask]
new_spks = twoP_data_truncated[:, mask]
new_dFF = dFF_data_truncated[:, mask]

# # Print final shapes to verify
# print("\nFinal shapes:")
# print(f"new_location: {new_location.shape}")
# print(f"new_spks: {new_spks.shape}")
# print(f"Number of time points kept: {np.sum(mask)}")

# Optional: Plot to verify
plt.figure(figsize=(12, 8))

plt.subplot(211)
plt.plot(new_location)
plt.title('New Location Data - inactive periods removed')
plt.xlabel('Time (frames)')
plt.ylabel('Location (cm)')

plt.subplot(212)
plt.plot(new_dFF[10,:])  # Plotting the third cell as an example
plt.title('New Spikes Data (example cell) - inactive periods removed')
plt.xlabel('Time (frames)')
plt.ylabel('dFF')

plt.tight_layout()
plt.show()

**4. Discretize spatial data (1cm/bin) and assign 2p data to a corresponding bin**

In [ ]:
# from helper import spatial_discretization as SD

# single_revolution_VR = 285.9317
# single_revolution_treadmill = 27.8
# single_lap_VR = 1146
# single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR
# # print(single_lap_treadmill)

# spks_3d, location_2d, lapnum = SD.reshape_into_laps(trimmed_twoP_data['smoothed_spks'], trimmed_VR_data['interp_location'])
# spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(lapnum, spks_3d, location_2d, single_lap_treadmill)

# #make a plot of the spatial activity of a single cell (each lap in a different row)
# plt.figure(figsize=(10, 5))
# plt.imshow(spatial_activity[15, :, :])
# plt.colorbar()
# plt.title('Spatial Activity of an example cell, #10')
# plt.xlabel('Distance (cm)')
# plt.ylabel('Trials')
# plt.clim([0, 1])
# plt.show()


In [ ]:
from helper import spatial_discretization as SD

single_revolution_VR = 285.9317
single_revolution_treadmill = 27.8
single_lap_VR = 1146
single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

# Then perform spatial assignment on the filtered data
spatial_activity, spatial_bins, trial_averaged_activity, bin_centers = SD.spatial_assignment(
    n_valid_laps,
    filtered_dFF_laps, 
    filtered_location_laps, 
    single_lap_treadmill
)

**5. Response profile (add all the spikes or average all the data points in each spatial bin to get the activity map)****

In [ ]:
# Plot the first 10 of the first dimension
for j in range(20):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Raw data
    ax1.set_title(f'Raw Data - Cell {j+1}')
    for i in range(20):
        ax1.plot(bin_centers, spatial_activity[j, i, :], label=f'Lap {i+1}', alpha=0.2)
    ax1.plot(bin_centers, trial_averaged_activity[j,:], label='Average', color='black', linewidth=2)
    ax1.set_xlabel('Position')
    ax1.set_ylabel('Activity')
    
    # Interpolated data
    ax2.set_title(f'Interpolated Data - Cell {j+1}')
    for i in range(20):
        ax2.plot(bin_centers, spatial_activity[j, i, :], label=f'Lap {i+1}', alpha=0.2)
    ax2.plot(bin_centers, trial_averaged_activity[j,:], label='Average', color='red', linewidth=2)
    ax2.set_xlabel('Position')
    ax2.set_ylabel('Activity')
    
    plt.tight_layout()
    plt.show()

# Also let's add a direct comparison of the differences:
for j in range(20):
    diff = np.abs(spatial_activity[j] - spatial_activity[j])
    print(f"Cell {j}: Max difference between raw and interpolated: {np.max(diff):.4f}")
    print(f"Cell {j}: Mean difference: {np.mean(diff):.4f}")

**6. Test for reliability for individual cells (calculate Pearson CC or cohen’s D)**

In [ ]:
from helper import ReliabilityTesting as RT
# Run the analysis
reliable_cells, avg_cc, cohens_d, iter_cc, normalized_activity = RT.test_cell_reliability(
    spatial_activity,
    n_shuffles=100,            
    cc_percentile=95,           
    cohen_threshold=1.5,        # You might want to adjust this
    min_cc_threshold=0.4,       # And this
    min_activity_threshold=0.1  # And possibly this
)
# Print summary
print(f"Found {np.sum(reliable_cells)} reliable cells out of {len(reliable_cells)}")
print(f"Mean correlation for reliable cells: {np.mean(avg_cc[reliable_cells]):.3f}")
print(f"Mean Cohen's D for reliable cells: {np.mean(cohens_d[reliable_cells]):.3f}")

normalized_spatial_activity = RT.normalize_spatial_activity(spatial_activity)

def normalize_spatial_activity_for_dFF(spatial_activity):
    n_cells, n_trials, n_bins = spatial_activity.shape
    normalized_data = np.zeros_like(spatial_activity)
    
    for cell in range(n_cells):
        # Find min and max across all trials for this cell
        cell_data = spatial_activity[cell, :, :]
        min_val = np.min(cell_data)
        max_val = np.max(cell_data)
        
        if max_val > min_val:  # Avoid division by zero
            normalized_data[cell, :, :] = (cell_data - min_val) / (max_val - min_val)
    
    return normalized_data

normalized_spatial_activity_acrossall = normalize_spatial_activity_for_dFF(spatial_activity)

def zscore_spatial_activity(spatial_activity):
    n_cells, n_trials, n_bins = spatial_activity.shape
    normalized_data = np.zeros_like(spatial_activity)
    
    for cell in range(n_cells):
        cell_data = spatial_activity[cell, :, :]
        mean_val = np.mean(cell_data)
        std_val = np.std(cell_data)
        
        if std_val > 0:  # Avoid division by zero
            normalized_data[cell, :, :] = (cell_data - mean_val) / std_val
    
    return normalized_data

normalized_spatial_activity_zscore = zscore_spatial_activity(spatial_activity)

def threshold_normalize_spatial_activity(spatial_activity, percentile=10):
    n_cells, n_trials, n_bins = spatial_activity.shape
    normalized_data = np.zeros_like(spatial_activity)
    
    for cell in range(n_cells):
        cell_data = spatial_activity[cell, :, :]
        baseline = np.percentile(cell_data, percentile)
        max_val = np.max(cell_data)
        
        if max_val > baseline:  # Avoid division by zero
            normalized_data[cell, :, :] = (cell_data - baseline) / (max_val - baseline)
            # Clip negative values to zero
            normalized_data[cell, :, :] = np.clip(normalized_data[cell, :, :], 0, 1)
    
    return normalized_data

normalized_spatial_activity_threshold = threshold_normalize_spatial_activity(spatial_activity)


In [ ]:
RT.plot_reliable_cells_activity(normalized_spatial_activity, reliable_cells, max_cells=5, avg_cc=avg_cc, cohen_d=cohens_d)
RT.plot_reliable_cells_activity(normalized_spatial_activity_acrossall, reliable_cells, max_cells=5, avg_cc=avg_cc, cohen_d=cohens_d)
RT.plot_reliable_cells_activity(normalized_spatial_activity_zscore, reliable_cells, max_cells=5,  avg_cc=avg_cc, cohen_d=cohens_d)
RT.plot_reliable_cells_activity(normalized_spatial_activity_threshold, reliable_cells, max_cells=5,  avg_cc=avg_cc, cohen_d=cohens_d)
plt.show()

In [ ]:
reliable_avg = {}

for i in np.where(reliable_cells)[0]:
    reliable_avg[i] = np.mean(spatial_activity[i, :, :], axis=0)

# plot average activity of first 25 reliable cells in a 5 x 5 subplots
fig, axs = plt.subplots(5, 5, figsize=(20, 20))
reliable_indices = list(reliable_avg.keys())[:25]
for idx, i in enumerate(reliable_indices):
    ax = axs[idx // 5, idx % 5]
    ax.plot(bin_centers, reliable_avg[i])
    ax.set_title(f'Cell {i}')
    ax.set_xlabel('Position')
    ax.set_ylabel('Activity')

# set a title over all plots
fig.suptitle('Average DFF of example reliable cells', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.98])  # Adjust the rect parameter to add space at the top
plt.show()

In [ ]:
RT.plot_reliable_cells_activity(normalized_spatial_activity, reliable_cells, max_cells=100, avg_cc=avg_cc, cohen_d=cohens_d)
RT.plot_reliable_cells_waterfall(normalized_spatial_activity, reliable_cells, max_cells=100)
plt.show()

# RT.plot_reliable_cells_activity(spatial_activity, reliable_cells, max_cells=np.sum(reliable_cells), avg_cc=avg_cc, cohen_d=cohens_d)
# RT.plot_reliable_cells_waterfall(spatial_activity, reliable_cells, max_cells=np.sum(reliable_cells))
# plt.show()

**7. Plotting activity of all responsive cells (cross validation – split trials in half)**

In [ ]:
from helper import ResponseVisualization as RV
fig1, sorted_indices = RV.create_response_plot(spatial_activity, reliable_cells,clim=(0, 10))  # Optional: manually set contrast limits for stronger effect

# # For waterfall visualization (alternative approach):
# fig2 = RV.create_waterfall_plot(
#     spatial_activity,
#     reliable_cells
# )

plt.show()

**8. Population vector cross-correlation**